# 🎨 RMBG 2.0 — AI Background Remover
Upload the `webbria` folder to Colab, set runtime to **T4 GPU**, and run all cells.

**You need:**
- [Hugging Face token](https://huggingface.co/settings/tokens) with access to [briaai/RMBG-2.0](https://huggingface.co/briaai/RMBG-2.0)
- Free [ngrok auth token](https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
# 1. Install dependencies
!pip install -q flask pyngrok transformers==4.45.2 kornia timm pillow

In [ ]:
# 2. Enter your tokens
from getpass import getpass

HF_TOKEN = getpass("Hugging Face Token: ")
NGROK_TOKEN = getpass("Ngrok Auth Token: ")
print("Tokens set!")

In [ ]:
# 3. Load RMBG 2.0 model
import torch
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

model = AutoModelForImageSegmentation.from_pretrained(
    'briaai/RMBG-2.0', trust_remote_code=True, token=HF_TOKEN
).eval().to(device)

transform_image = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("Model loaded!")

In [ ]:
# 4. Launch Flask + ngrok
import os, io, uuid, threading
from flask import Flask, render_template, request, jsonify, send_file
from PIL import Image
from pyngrok import ngrok, conf

app = Flask(__name__,
            template_folder='templates',
            static_folder='static')
app.config['MAX_CONTENT_LENGTH'] = 20 * 1024 * 1024

ALLOWED_EXT = {'png','jpg','jpeg','webp','bmp','tiff'}

def remove_background(image):
    original = image.convert('RGB')
    inp = transform_image(original).unsqueeze(0).to(device)
    with torch.no_grad():
        preds = model(inp)[-1].sigmoid().cpu()
    mask = transforms.ToPILImage()(preds[0].squeeze()).resize(original.size)
    result = image.convert('RGBA')
    result.putalpha(mask)
    return result

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/remove-bg', methods=['POST'])
def remove_bg():
    if 'image' not in request.files:
        return jsonify({'error': 'No image provided.'}), 400
    f = request.files['image']
    if not f.filename or not ('.' in f.filename and f.filename.rsplit('.',1)[1].lower() in ALLOWED_EXT):
        return jsonify({'error': 'Unsupported file type.'}), 400
    try:
        result = remove_background(Image.open(f.stream))
        buf = io.BytesIO()
        result.save(buf, format='PNG')
        buf.seek(0)
        return send_file(buf, mimetype='image/png', download_name=f'rmbg_{uuid.uuid4().hex[:8]}.png')
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Start ngrok
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(5000)
print(f"\n{'='*50}")
print(f"  Your app is live at: {public_url}")
print(f"{'='*50}\n")

# Run Flask in background thread
threading.Thread(target=lambda: app.run(port=5000, use_reloader=False), daemon=True).start()